# Process ACS data

Outcomes:
- Load and process ACS data obtained from data.census.gov
- This example provides two preset functions for processing income and age groups

In [ ]:
import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2
from helpers import acs

In [ ]:
# ======================
# PATHS
# ======================
DATA_DIR = "data/acs"
OUTPUT_DIR = "data/processed"

# ACS table files
CBSA_CODE = 38060
INCOME_FILE = f"{DATA_DIR}/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B19001-Data.csv"   # household income
AGE_FILE    = f"{DATA_DIR}/ACS5Y2020/{CBSA_CODE}/ACSDT5Y2020.B01001-Data.csv"   # sex by age (individuals)

# outputs
OUT_PATH_ACS_DISTR      = f"{OUTPUT_DIR}/acs_cbg_distr_{CBSA_CODE}.csv"
OUT_PATH_INCOME_MARGINS = f"{OUTPUT_DIR}/acs_income_margins_{CBSA_CODE}.csv"
OUT_PATH_AGE_MARGINS    = f"{OUTPUT_DIR}/acs_age_margins_{CBSA_CODE}.csv"

# ======================
# INCOME GROUPING (B19001)
# (use quartiles obtained from acs.compute_income_quartile_mapping
# or specify your own group mapping; if specify, the group labels/keys
# should match the format below for downstream processing)
# ======================

INCOME_GROUPS = acs.compute_income_quartile_mapping(INCOME_FILE)

"""
INCOME_GROUPS = {
    "<35k": [
        "B19001_002E", 
        "B19001_003E", 
        "B19001_004E",
        "B19001_005E", 
        "B19001_006E", 
        "B19001_007E",
    ],

    "35k-75k": [
        "B19001_008E", 
        "B19001_009E",
        "B19001_010E", 
        "B19001_011E", 
        "B19001_012E",
    ],

    "75k-100k": [
        "B19001_013E",
    ],

    "100k+": [
        "B19001_014E", 
        "B19001_015E",
        "B19001_016E", 
        "B19001_017E",
    ],
}
"""

# ======================
# AGE GROUPING (B01001)
# The mobility sample is 18+, so <18 is dropped below (DROP_AGE_GROUPS).
# Group labels/order must match the downstream calibration targets.
# ======================

AGE_GROUPS = {

    "<18": [
        "Under 5 years", 
        "5 to 9 years",
        "10 to 14 years", 
        "15 to 17 years",
    ],

    "18-24": [
        "18 and 19 years", 
        "20 years",
        "21 years", 
        "22 to 24 years",
    ],

    "25-44": [
        "25 to 29 years", 
        "30 to 34 years",
        "35 to 39 years", 
        "40 to 44 years",
    ],

    "45-64": [
        "45 to 49 years",
        "50 to 54 years",
        "55 to 59 years",
        "60 and 61 years",
        "62 to 64 years",
    ],

    "65+": [
        "65 and 66 years",
        "67 to 69 years",
        "70 to 74 years",
        "75 to 79 years",
        "80 to 84 years",
        "85 years and over",
    ],
}

# drop groups (e.g. if mobility sample only contains 18+ users)
DROP_AGE_GROUPS = ["<18"]

# ======================
# NOTES
# ======================
# - Paths assume CSV downloads from data.census.gov
# - skiprows behavior handled in processing functions
# - order of group labels must align with downstream calibration targets

In [ ]:
INCOME_GROUPS

In [ ]:
income_cbg, income_cbsa = acs.process_income_table(INCOME_FILE, INCOME_GROUPS, cbsa_code=CBSA_CODE)
income_cbsa = acs.format_cbsa_marginals(income_cbsa, var_name="hh_income")

In [ ]:
income_cbg

In [ ]:
income_cbsa # target marginals for the CBSA

In [ ]:
age_cbg, age_cbsa = acs.process_age_table(AGE_FILE, AGE_GROUPS, DROP_AGE_GROUPS, cbsa_code=CBSA_CODE)
age_cbsa = acs.format_cbsa_marginals(age_cbsa, var_name="age_group")

In [ ]:
age_cbg

In [ ]:
age_cbsa

In [ ]:
cbg_distr = acs.merge_distr_tables([age_cbg, income_cbg], on='GEOID', how='inner', dropna=True)
cbg_distr

In [ ]:
cbg_distr.to_csv(OUT_PATH_ACS_DISTR, index=False)
income_cbsa.to_csv(OUT_PATH_INCOME_MARGINS, index=False)
age_cbsa.to_csv(OUT_PATH_AGE_MARGINS, index=False)